## Why Dataset Quality Matters

A machine learning model learns from the data we provide.

Problems such as corrupted images, incorrect labels, duplicate images,
or data leakage can negatively affect training and can produce misleading
evaluation results.

Therefore, dataset quality is checked before model training.

## How Is Pixel Data Extracted from an Image?

To detect exact duplicate images, we need a way to compare the actual content of images rather than their filenames.

For example, these two files may have different filenames:

```text
train/horse/horse_001.jpg
test/horse/image_9283.jpg
```

The filenames tell us nothing about whether the images contain the same visual content.

Instead, we compare the **pixel data** that makes up each image.

---

## 1. What Is a Pixel?

A digital image is made up of many small points called **pixels**.

Each pixel contains numerical information that represents its color.

For an RGB image, every pixel contains three values:

- **R** → Red
- **G** → Green
- **B** → Blue

Each value normally ranges from:

```text
0 → 255
```

For example:

```text
[255, 0, 0]
```

represents a red pixel.

Similarly:

```text
[0, 255, 0]
```

represents a green pixel, and:

```text
[0, 0, 255]
```

represents a blue pixel.

Therefore, an RGB image can be represented as a large collection of numerical pixel values.

---

## 2. Image Dimensions and Pixel Data

Suppose an image has dimensions:

```text
256 × 256
```

This means the image contains:

```text
256 × 256 = 65,536 pixels
```

Since it is an RGB image, each pixel has three color values:

```text
Red
Green
Blue
```

Therefore, conceptually, the image can be represented as:

```text
Height × Width × Channels

256 × 256 × 3
```

The three channels correspond to:

```text
Channel 1 → Red
Channel 2 → Green
Channel 3 → Blue
```

So an image is ultimately represented by a large collection of numerical values.

---

## 3. Opening the Image with PIL

In Python, we use the **Pillow (PIL)** library to read the image.

```python
from PIL import Image

img = Image.open(image_path)
```

`Image.open()` reads the image file and creates a PIL Image object.

For example:

```python
img = Image.open("horse.jpg")
```

The resulting `img` object contains information about the image, including:

- Width
- Height
- Image format
- Color mode
- Pixel information

---

## 4. Converting the Image to RGB

Our dataset contains images with different color modes.

During dataset inspection, we found modes such as:

```text
RGB
RGBA
CMYK
L
P
```

These representations are different.

For example:

### RGB

Contains:

```text
Red
Green
Blue
```

### RGBA

Contains:

```text
Red
Green
Blue
Alpha
```

The Alpha channel represents transparency.

### Grayscale (`L`)

Contains only one intensity value per pixel rather than three RGB values.

### CMYK

Uses:

```text
Cyan
Magenta
Yellow
Black
```

instead of RGB.

To make the comparison consistent, we convert every image to RGB:

```python
img = img.convert("RGB")
```

After this conversion, every image has the same basic representation:

```text
Red + Green + Blue
```

This is important because we want the hashing process to work with a standardized representation.

---

## 5. Extracting the Pixel Data

Once the image has been converted to RGB, we can access its raw pixel representation.

We use:

```python
pixel_data = img.tobytes()
```

The `tobytes()` method converts the image's pixel information into a sequence of raw bytes.

Conceptually, the process is:

```text
Image
   ↓
PIL Image Object
   ↓
RGB Pixel Values
   ↓
tobytes()
   ↓
Raw Pixel Bytes
```

For a very small RGB image, the pixel values might conceptually look like:

```text
Pixel 1 → [255, 0, 0]
Pixel 2 → [0, 255, 0]
Pixel 3 → [0, 0, 255]
```

After converting the pixel information to bytes, Python obtains a binary representation of those values.

We do not need to manually inspect or print these bytes.

Their purpose is to provide a consistent representation of the actual image content that can be passed to the hashing algorithm.

---

## 6. Why Use `tobytes()`?

The goal of this step is not to display the pixels.

The goal is to create a representation that can be given to a hashing algorithm.

We can think of it as:

```text
Image
   ↓
Pixel information
   ↓
Raw bytes
   ↓
Hash function
   ↓
Fingerprint
```

The hash function takes the large amount of pixel information and produces a much shorter value called a **hash**.

---

## 7. Generating an MD5 Hash

After obtaining the pixel bytes, we generate an MD5 hash:

```python
import hashlib

image_hash = hashlib.md5(pixel_data).hexdigest()
```

The complete operation is:

```python
image_hash = hashlib.md5(img.tobytes()).hexdigest()
```

The result is a hexadecimal string such as:

```text
9f86d081884c7d659a2feaa0c55ad015
```

This hash can be treated as a **fingerprint of the image's pixel data**.

Instead of comparing thousands or millions of pixel values directly, we can compare these fingerprints.

---

## 8. Example of Exact Duplicate Detection

Suppose we have two image files:

```text
train/cat/cat_001.jpg
test/cat/cat_837.jpg
```

If both files contain exactly the same image, after RGB conversion their pixel data will be identical.

Therefore:

```text
Image A
   ↓
RGB pixel data
   ↓
MD5
   ↓
ABC123...

Image B
   ↓
RGB pixel data
   ↓
MD5
   ↓
ABC123...
```

The hashes are identical.

Therefore, we identify the images as **exact duplicates**.

The filenames do not matter.

The folders do not matter.

The important factor is that the standardized pixel data produces the same hash.

---

## 9. Why This Works Even When File Names Are Different

Consider:

```text
train/dog/photo_001.jpg
test/dog/IMG_9283.jpg
```

The names are completely different.

However, if both files contain exactly the same pixel information:

```text
photo_001.jpg
       ↓
RGB pixel data
       ↓
MD5 → ABC123

IMG_9283.jpg
       ↓
RGB pixel data
       ↓
MD5 → ABC123
```

The matching hashes reveal that the images contain identical pixel data.

Therefore, duplicate detection is based on **image content rather than filenames**.

---

## 10. Why We Convert to RGB Before Hashing

Standardization is important.

Suppose two files represent the same visual image but one is stored as RGB and another uses a different image mode.

If we directly compare their raw representations, they may not be represented in the same way.

By doing:

```python
img = img.convert("RGB")
```

before:

```python
img.tobytes()
```

we make the pixel representation consistent.

The process becomes:

```text
Original Image
      ↓
Open with PIL
      ↓
Convert to RGB
      ↓
Extract pixel bytes
      ↓
Generate MD5 hash
```

This gives us a standardized basis for exact duplicate comparison.

---

## 11. What Our Script Reports

Our duplicate-detection script records the hash associated with each image.

Conceptually:

```text
Hash                         Image
------------------------------------------------
ABC123...                    train/cat/img1.jpg
XYZ789...                    train/horse/img2.jpg
ABC123...                    test/cat/img9.jpg
```

When a hash appears more than once, the corresponding images are potential exact duplicates.

We then specifically check whether the duplicate images occur in different dataset splits.

For example:

```text
Same hash
   ↓
train + test
   ↓
Cross-split duplicate
```

This is more important than a duplicate occurring twice inside the training set.

---

## Summary of the Process

The complete exact-duplicate detection process is:

```text
                    IMAGE
                      │
                      ▼
              Open with PIL
                      │
                      ▼
               Convert to RGB
                      │
                      ▼
              Extract pixel data
                 img.tobytes()
                      │
                      ▼
               Generate MD5
                      │
                      ▼
              Image fingerprint
                      │
                      ▼
              Compare fingerprints
                      │
             ┌────────┴────────┐
             │                 │
          Same hash         Different hash
             │                 │
             ▼                 ▼
      Exact duplicate      Not an exact
                           duplicate
```

In [ ]:
from pathlib import Path
from PIL import Image
import hashlib

In [ ]:
DATA_DIR = Path("../data")

splits = ["train", "test", "validation"]

def get_image_hash(image_path):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        return hashlib.md5(img.tobytes()).hexdigest()

hashes = {}

for split in splits:
    print(f"\nChecking {split}...")

    for class_path in (DATA_DIR / split).iterdir():
        if not class_path.is_dir():
            continue

        for image_path in class_path.iterdir():
            if image_path.is_file():
                try:
                    image_hash = get_image_hash(image_path)

                    if image_hash not in hashes:
                        hashes[image_hash] = []

                    hashes[image_hash].append(
                        (split, class_path.name, str(image_path))
                    )

                except Exception:
                    pass


In [ ]:
duplicates = {
    h: files
    for h, files in hashes.items()
    if len(files) > 1
}

print("\n" + "=" * 50)
print("Duplicate Images:")
print("=" * 50)

print("Unique images:", len(hashes))
print("Duplicate groups:", len(duplicates))

cross_split_duplicates = []

for image_hash, files in duplicates.items():
    split_names = set(item[0] for item in files)

    if len(split_names) > 1:
        cross_split_duplicates.append(files)

print("Cross-split duplicate groups:", len(cross_split_duplicates))

if cross_split_duplicates:
    print("\nExamples:")

    for group in cross_split_duplicates[:10]:
        print("\n---")
        for split, class_name, path in group:
            print(f"{split:12} | {class_name:10} | {path}")



Duplicate Images:
Unique images: 10726
Duplicate groups: 52
Cross-split duplicate groups: 15

Examples:

---
train        | Cat        | ..\data\train\Cat\00000302_007.jpg
test         | Cat        | ..\data\test\Cat\00000298_014.jpg

---
train        | Chicken    | ..\data\train\Chicken\photo-1556316918-880f9e893822_ixlib=rb-4.0.jpg
test         | Chicken    | ..\data\test\Chicken\five brown hens on ground beside fence.jpg

---
train        | Donkey     | ..\data\train\Donkey\216793810.jpg
validation   | Donkey     | ..\data\validation\Donkey\240_F_216793810_X7iWodtAKSib4rPvoW4sDQDp2MXVoZhC.jpg

---
train        | Donkey     | ..\data\train\Donkey\352826764.jpg
test         | Donkey     | ..\data\test\Donkey\240_F_352826764_n9ZlT2RKxweXsXvTGZReGH737ytQQzm3.jpg

---
train        | Donkey     | ..\data\train\Donkey\Donkey_in_Clovelly,_North_Devon,_England.jpg
test         | Donkey     | ..\data\test\Donkey\640px-Donkey_in_Clovelly_2C_North_Devon_2C_England.jpg

---
train        | Goat 

## After Manually Removing the Cross-split duplicate groups: 15

In [9]:
DATA_DIR = Path("../data")

splits = ["train", "test", "validation"]

def get_image_hash(image_path):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        return hashlib.md5(img.tobytes()).hexdigest()

hashes = {}

for split in splits:
    print(f"\nChecking {split}...")

    for class_path in (DATA_DIR / split).iterdir():
        if not class_path.is_dir():
            continue

        for image_path in class_path.iterdir():
            if image_path.is_file():
                try:
                    image_hash = get_image_hash(image_path)

                    if image_hash not in hashes:
                        hashes[image_hash] = []

                    hashes[image_hash].append(
                        (split, class_path.name, str(image_path))
                    )

                except Exception:
                    pass



Checking train...

Checking test...

Checking validation...


In [10]:
duplicates = {
    h: files
    for h, files in hashes.items()
    if len(files) > 1
}

print("\n" + "=" * 50)
print("Duplicate Images:")
print("=" * 50)

print("Unique images:", len(hashes))
print("Duplicate groups:", len(duplicates))

cross_split_duplicates = []

for image_hash, files in duplicates.items():
    split_names = set(item[0] for item in files)

    if len(split_names) > 1:
        cross_split_duplicates.append(files)

print("Cross-split duplicate groups:", len(cross_split_duplicates))

if cross_split_duplicates:
    print("\nExamples:")

    for group in cross_split_duplicates[:10]:
        print("\n---")
        for split, class_name, path in group:
            print(f"{split:12} | {class_name:10} | {path}")



Duplicate Images:
Unique images: 10723
Duplicate groups: 37
Cross-split duplicate groups: 0


## Results

After running the exact duplicate detection script on the cleaned dataset, the following results were obtained:


### Duplicate Images:

Unique images: 10723
Duplicate groups: 37
Cross-split duplicate groups: 0

Interpretation

The dataset currently contains 10,723 unique image hashes.

The script identified 37 duplicate groups. These duplicate groups exist within the dataset but do not occur across different dataset splits.

Most importantly:

Cross-split duplicate groups: 0

This means that no exact pixel-level duplicate image was found between the:

Training set

Validation set

Test set

Therefore, based on the exact duplicate check, there is no detected exact-duplicate data leakage between the dataset splits.